# 📈 Capstone 9.4: Model Comparison

**Goal**: Compare all trained models, visualize results, and select the champion.

---

In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.autolog(disable=True)
print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Gather All Runs

In [ ]:
# Get all runs (excluding nested HPO trials)
all_runs = mlflow.search_runs(
    experiment_names=["capstone-wine-quality"],
    order_by=["metrics.accuracy DESC"]
)

# Filter out nested runs (HPO trials) — keep only parent/top-level runs with metrics
# Nested runs have 'mlflow.parentRunId' tag
top_level_runs = all_runs[
    all_runs.get("tags.mlflow.parentRunId", pd.Series([None]*len(all_runs))).isna()
    & all_runs["metrics.accuracy"].notna()
].copy()

print(f"Total runs: {len(all_runs)}")
print(f"Top-level runs with metrics: {len(top_level_runs)}")

## 2. Comparison Table

In [ ]:
# Build clean comparison DataFrame
comparison = []
for _, run in top_level_runs.iterrows():
    comparison.append({
        "Run Name": run.get("tags.mlflow.runName", "unnamed"),
        "Model Type": run.get("tags.model_type", "unknown"),
        "Stage": run.get("tags.stage", "-"),
        "Accuracy": run.get("metrics.accuracy"),
        "F1 Weighted": run.get("metrics.f1_weighted"),
        "Run ID": run["run_id"][:8] + "...",
        "Full Run ID": run["run_id"],
    })

comp_df = pd.DataFrame(comparison).sort_values("Accuracy", ascending=False)
print("📊 Model Comparison:")
print("=" * 90)
display_df = comp_df.drop(columns=["Full Run ID"])
print(display_df.to_string(index=False))

## 3. Visualize Comparison

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_df = comp_df.head(8)  # Top 8 runs

# Accuracy
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(plot_df))]
axes[0].barh(plot_df["Run Name"], plot_df["Accuracy"], color=colors)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Model Accuracy Comparison")
axes[0].invert_yaxis()
for i, v in enumerate(plot_df["Accuracy"]):
    if pd.notna(v):
        axes[0].text(v + 0.001, i, f"{v:.4f}", va='center')

# F1 Score
if plot_df["F1 Weighted"].notna().any():
    axes[1].barh(plot_df["Run Name"], plot_df["F1 Weighted"].fillna(0), color=colors)
    axes[1].set_xlabel("F1 Score (weighted)")
    axes[1].set_title("Model F1 Score Comparison")
    axes[1].invert_yaxis()
    for i, v in enumerate(plot_df["F1 Weighted"]):
        if pd.notna(v):
            axes[1].text(v + 0.001, i, f"{v:.4f}", va='center')

plt.tight_layout()
plt.show()

## 4. Select Champion & Challenger

In [ ]:
# Best run
champion_row = comp_df.iloc[0]
champion_run_id = champion_row["Full Run ID"]

print(f"🏆 CHAMPION: {champion_row['Run Name']}")
print(f"   Type: {champion_row['Model Type']}")
print(f"   Accuracy: {champion_row['Accuracy']:.4f}")
print(f"   Run ID: {champion_run_id}")

# Second best
if len(comp_df) >= 2:
    challenger_row = comp_df.iloc[1]
    challenger_run_id = challenger_row["Full Run ID"]
    print(f"\n⚔️  CHALLENGER: {challenger_row['Run Name']}")
    print(f"   Type: {challenger_row['Model Type']}")
    print(f"   Accuracy: {challenger_row['Accuracy']:.4f}")
    print(f"   Run ID: {challenger_run_id}")

print(f"\n📝 Save these Run IDs for the next notebook!")
print(f"   champion_run_id = \"{champion_run_id}\"")
if len(comp_df) >= 2:
    print(f"   challenger_run_id = \"{challenger_run_id}\"")

## 5. Log Comparison Artifacts

In [ ]:
with mlflow.start_run(run_name="model_comparison", experiment_id=mlflow.get_experiment_by_name("capstone-wine-quality").experiment_id):
    mlflow.set_tag("stage", "comparison")
    mlflow.set_tag("author", "sujat")
    
    # Log comparison table
    mlflow.log_text(
        comp_df.drop(columns=["Full Run ID"]).to_string(index=False),
        "comparison_table.txt"
    )
    
    # Log best model info
    mlflow.log_params({
        "champion_model": champion_row["Run Name"],
        "champion_accuracy": champion_row["Accuracy"],
    })
    
    # Save comparison chart
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(plot_df))]
    ax.barh(plot_df["Run Name"], plot_df["Accuracy"], color=colors)
    ax.set_xlabel("Accuracy")
    ax.set_title("Capstone — Model Comparison")
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig("comparison_chart.png", dpi=150)
    mlflow.log_artifact("comparison_chart.png", "plots")
    plt.close()
    import os; os.remove("comparison_chart.png")
    
    print("✅ Comparison artifacts logged!")

## ➡️ Next: `05_registry_workflow.ipynb` — Register and manage the champion